# Solar filament segmentation: reproduce the final predictions

Runs the submitted system (public leaderboard PQ 0.40) on the competition test images and writes `submission.csv`.
Everything it needs is in this repository: code and all model weights (`final/weights`, about 400 MB, float16).
No other file or download is required, except the test images themselves and the pip packages.

System: four detectors (Mask R-CNN at 2048 px, YOLO11m-seg, RT-DETR-l, YOLO11l-seg) run on each image and its mirror;
a crop-level U-Net redraws every proposal; a ConvNeXt-Tiny U-Net adds instances from a semantic segmentation;
per-source isotonic calibration; a score-weighted pixel vote over overlapping masks gives the final masks.

In [ ]:
# run from the repository root
import os, subprocess, sys
if os.path.basename(os.getcwd()) == "final":
    os.chdir("..")
print(os.getcwd())
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])  # once, on a fresh machine

In [ ]:
# folder with the competition test images (180 GONG .jpeg files)
IMAGES = "data/MAGFiLO_1.0_Kaggle_2026/test/test_images"
LIMIT = 0   # set to e.g. 5 for a quick check; 0 = all images. About 30 s per image on an RTX 4060, slower on CPU
assert os.path.isdir(IMAGES), "put the test images in " + IMAGES + " or edit IMAGES"
cmd = [sys.executable, "final/predict.py", "--images", IMAGES, "--out", "submission.csv"] + (["--limit", str(LIMIT)] if LIMIT else [])
subprocess.check_call(cmd)

In [ ]:
# format check: one row per filament, ids <image stem>_<k>, run-length-encoded masks that do not overlap
import numpy as np, pandas as pd, pycocotools.mask as mu
df = pd.read_csv("submission.csv")
print(len(df), "filaments in", df.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
for stem, g in df.groupby(df.filament_id.str.rsplit("_", n=1).str[0]):
    m = np.stack([mu.decode({"size": [2048, 2048], "counts": r.encode()}) for r in g.segmentation_rle])
    assert m.sum(0).max() <= 1, stem + ": overlapping masks"
print("ok: masks are disjoint in every image")